# Grok-multimodal · FS00-FS02 Foundations

**Domain**: multimodal · **Stages**: from-scratch base

| Stage | Topic |
|-------|-------|
| FS00 | Modality / alignment / task map |
| FS01 | Image tensor + bag-of-colors pseudo-caption |
| FS02 | Mini CNN image to class word |

Each stage: concept, minimal impl, real input, output, compare.


In [ ]:
import os, json, math, random, time, platform
from pathlib import Path
import numpy as np

os.environ.pop("CUDA_VISIBLE_DEVICES", None)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT = Path("/kaggle/working")
FIG = OUT / "figures"; RES = OUT / "results"
FIG.mkdir(parents=True, exist_ok=True); RES.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "device", device)
print("gpus", torch.cuda.device_count() if torch.cuda.is_available() else 0)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(" ", i, torch.cuda.get_device_name(i))
PROGRESS = {}


## FS00 · Modalities, alignment, task map

**Concept**: modalities as tensors; alignment = comparable score; tasks = encode / retrieve / condition language / condition media / route any-to-any.

No training here — synthetic image + toy text distances.


In [ ]:
def make_shape_image(kind: str, size=64):
    img = np.ones((size, size, 3), dtype=np.float32) * 0.95
    yy, xx = np.mgrid[0:size, 0:size]
    cy, cx = size // 2, size // 2
    if kind == "red_circle":
        mask = (yy - cy) ** 2 + (xx - cx) ** 2 <= (size * 0.28) ** 2
        img[mask] = (0.9, 0.15, 0.12)
    elif kind == "blue_square":
        m = (np.abs(yy - cy) < size * 0.25) & (np.abs(xx - cx) < size * 0.25)
        img[m] = (0.15, 0.25, 0.85)
    elif kind == "green_triangle":
        m = (yy > cy - size * 0.25) & (yy < cy + size * 0.3)
        m &= np.abs(xx - cx) < (yy - (cy - size * 0.25)) * 0.7
        img[m] = (0.15, 0.75, 0.25)
    else:
        raise ValueError(kind)
    return img

def text_embed(s: str, dim=32):
    v = np.zeros(dim, dtype=np.float32)
    for i, ch in enumerate(s.lower()):
        v[ord(ch) % dim] += 1.0
        v[(ord(ch) * 7 + i) % dim] += 0.3
    n = np.linalg.norm(v) + 1e-8
    return v / n

def image_color_embed(img, dim=32):
    h, w, _ = img.shape
    feats = [img.mean(axis=(0, 1))]
    for by in range(2):
        for bx in range(2):
            patch = img[by*h//2:(by+1)*h//2, bx*w//2:(bx+1)*w//2]
            feats.append(patch.mean(axis=(0, 1)))
    raw = np.concatenate(feats)
    out = np.zeros(dim, dtype=np.float32)
    out[: min(dim, len(raw))] = raw[:dim]
    out[15] = img[..., 0].mean(); out[16] = img[..., 1].mean(); out[17] = img[..., 2].mean()
    n = np.linalg.norm(out) + 1e-8
    return out / n

pairs = [
    ("red_circle", "a red circle"),
    ("blue_square", "a blue square"),
    ("green_triangle", "a green triangle"),
    ("red_circle", "a blue square"),
]
rows = []
for kind, txt in pairs:
    img = make_shape_image(kind)
    ie, te = image_color_embed(img), text_embed(txt)
    sim = float(np.dot(ie, te))
    color_word = {"red": 0, "blue": 1, "green": 2}
    shape_word = {"circle": 0, "square": 1, "triangle": 2}
    img_color = {"red_circle": 0, "blue_square": 1, "green_triangle": 2}[kind]
    img_shape = {"red_circle": 0, "blue_square": 1, "green_triangle": 2}[kind]
    tw = txt.split()
    t_color = next((color_word[w] for w in tw if w in color_word), -1)
    t_shape = next((shape_word[w] for w in tw if w in shape_word), -1)
    align = int(t_color == img_color) + int(t_shape == img_shape)
    rows.append({"image": kind, "text": txt, "dot_embed": round(sim, 4), "hand_align_0_2": align, "match": align == 2})

print("FS00 alignment table:")
for r in rows:
    print(r)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, kind in zip(axes, ["red_circle", "blue_square", "green_triangle"]):
    ax.imshow(make_shape_image(kind))
    ax.set_title(kind); ax.axis("off")
fig.suptitle("FS00 synthetic modality inputs")
fig.tight_layout()
fig.savefig(FIG / "fs00_modalities.png", dpi=120)
plt.close()
print("wrote", FIG / "fs00_modalities.png")

fs00 = {
    "stage": "FS00",
    "concept": "modalities as tensors + alignment as comparable score",
    "rows": rows,
    "figure": "figures/fs00_modalities.png",
    "takeaway": "matched pairs hand_align=2, mismatches <2",
}
(RES / "fs00.json").write_text(json.dumps(fs00, indent=2, ensure_ascii=False))
PROGRESS["FS00"] = "ok"
print("FS00 DONE")


## FS01 · Bag-of-colors captioner

**Concept**: dumb but runnable image-to-text via color histogram + shape heuristics.

**vs FS00**: FS00 only scores pairs; FS01 *generates* a phrase.


In [ ]:
COLOR_NAMES = [
    ((0.55, 1.0), (0.0, 0.45), (0.0, 0.45), "red"),
    ((0.0, 0.45), (0.0, 0.5), (0.45, 1.0), "blue"),
    ((0.0, 0.55), (0.45, 1.0), (0.0, 0.55), "green"),
]

def bag_of_colors_caption(img):
    h, w, _ = img.shape
    crop = img[h//4:3*h//4, w//4:3*w//4]
    mean = crop.mean(axis=(0, 1))
    # nearest named color by L2 in RGB
    centers = {"red": np.array([0.9,0.15,0.12]), "blue": np.array([0.15,0.25,0.85]), "green": np.array([0.15,0.75,0.25])}
    name = min(centers, key=lambda n: np.linalg.norm(mean - centers[n]))
    bg = img.mean(axis=2) > 0.9
    ink = ~bg
    if ink.sum() < 10:
        shape = "blank"
    else:
        ys, xs = np.where(ink)
        bw = xs.max() - xs.min() + 1
        bh = ys.max() - ys.min() + 1
        aspect = bw / max(1, bh)
        fill = ink.sum() / max(1, bw * bh)
        # circle has medium fill; square high; triangle lower fill / pointy
        if fill >= 0.78:
            shape = "square"
        elif fill >= 0.55:
            shape = "circle"
        else:
            shape = "triangle"
    return f"a {name} {shape}", mean.tolist(), shape

inputs = ["red_circle", "blue_square", "green_triangle"]
fs01_rows = []
fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for ax, kind in zip(axes, inputs):
    img = make_shape_image(kind)
    cap, mean, shape = bag_of_colors_caption(img)
    gt = "a " + kind.replace("_", " ")
    ok = cap == gt
    fs01_rows.append({"input": kind, "gt": gt, "pred": cap, "exact_match": ok, "mean_rgb": [round(x,3) for x in mean]})
    ax.imshow(img); ax.set_title(f"pred: {cap}\nGT: {gt}", fontsize=9); ax.axis("off")
fig.suptitle("FS01 bag-of-colors captioner")
fig.tight_layout(); fig.savefig(FIG / "fs01_captions.png", dpi=120); plt.close()
print(json.dumps(fs01_rows, indent=2))
acc = sum(r["exact_match"] for r in fs01_rows) / len(fs01_rows)
print("FS01 exact_match_acc", acc)
fs01 = {
    "stage": "FS01",
    "method": "bag-of-colors + geometry heuristics",
    "exact_match_acc": acc,
    "rows": fs01_rows,
    "vs_prev": "FS00 only scored pairs; FS01 emits captions",
    "figure": "figures/fs01_captions.png",
}
(RES / "fs01.json").write_text(json.dumps(fs01, indent=2, ensure_ascii=False))
PROGRESS["FS01"] = "ok"
print("FS01 DONE")


## FS02 · Tiny CNN to class word

**Concept**: learnable conv replaces rules (ImageNet-era backbone idea, mini).

**vs FS01**: rules break OOD; CNN fits decision boundary on train distribution.


In [ ]:
CLASSES = ["red_circle", "blue_square", "green_triangle"]
class_to_id = {c: i for i, c in enumerate(CLASSES)}

def synth_dataset(n_per=200, size=32, noise=0.05):
    xs, ys = [], []
    for c in CLASSES:
        for _ in range(n_per):
            img = make_shape_image(c, size=64)
            img = img[::2, ::2][:size, :size]
            img = np.clip(img + np.random.randn(*img.shape).astype(np.float32) * noise, 0, 1)
            xs.append(img.transpose(2, 0, 1))
            ys.append(class_to_id[c])
    x = torch.tensor(np.stack(xs), dtype=torch.float32)
    y = torch.tensor(ys, dtype=torch.long)
    return x, y

class TinyCNN(nn.Module):
    def __init__(self, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
            nn.Flatten(), nn.Linear(64, n_classes),
        )
    def forward(self, x):
        return self.net(x)

x_all, y_all = synth_dataset()
n = len(x_all)
perm = torch.randperm(n)
x_all, y_all = x_all[perm], y_all[perm]
n_train = int(0.8 * n)
train_loader = DataLoader(TensorDataset(x_all[:n_train], y_all[:n_train]), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(x_all[n_train:], y_all[n_train:]), batch_size=128)

model = TinyCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
hist = []
t0 = time.time()
for epoch in range(1, 8):
    model.train(); tl, nseen = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model(xb), yb)
        loss.backward(); opt.step()
        tl += loss.item() * xb.size(0); nseen += xb.size(0)
    model.eval(); correct = total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(1)
            correct += (pred == yb).sum().item(); total += yb.size(0)
    row = {"epoch": epoch, "train_loss": round(tl/nseen, 4), "val_acc": round(correct/total, 4)}
    hist.append(row); print(row)
train_s = round(time.time() - t0, 3)

def cnn_caption(kind):
    img = make_shape_image(kind, 64)[::2, ::2][:32, :32]
    t = torch.tensor(img.transpose(2,0,1)[None], dtype=torch.float32, device=device)
    with torch.no_grad():
        pred = model(t).argmax(1).item()
    word = CLASSES[pred].replace("_", " ")
    return f"a {word}"

cmp_rows = []
for kind in CLASSES:
    rule_cap, _, _ = bag_of_colors_caption(make_shape_image(kind))
    cnn_cap = cnn_caption(kind)
    cmp_rows.append({"input": kind, "fs01_rule": rule_cap, "fs02_cnn": cnn_cap})

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for ax, kind in zip(axes, CLASSES):
    ax.imshow(make_shape_image(kind))
    ax.set_title(f"CNN: {cnn_caption(kind)}", fontsize=9); ax.axis("off")
fig.suptitle("FS02 CNN image to word")
fig.tight_layout(); fig.savefig(FIG / "fs02_cnn_words.png", dpi=120); plt.close()

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot([h["epoch"] for h in hist], [h["val_acc"] for h in hist], marker="o")
ax.set_xlabel("epoch"); ax.set_ylabel("val_acc"); ax.set_title("FS02 learning curve"); ax.set_ylim(0, 1.05)
fig.tight_layout(); fig.savefig(FIG / "fs02_curve.png", dpi=120); plt.close()

fs02 = {
    "stage": "FS02",
    "method": "TinyCNN classifier to class name string",
    "val_acc": hist[-1]["val_acc"],
    "history": hist,
    "train_seconds": train_s,
    "comparison_rows": cmp_rows,
    "vs_prev": "FS01 fixed rules; FS02 learns filters (closed-set labels, not open captions)",
    "figures": ["figures/fs02_cnn_words.png", "figures/fs02_curve.png"],
}
(RES / "fs02.json").write_text(json.dumps(fs02, indent=2, ensure_ascii=False))
PROGRESS["FS02"] = "ok"
print("FS02 DONE", fs02["val_acc"])


In [ ]:
summary = {
    "notebook": "Grok-multimodal-fs00-fs02-foundations",
    "device": str(device),
    "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
    "progress": PROGRESS,
    "chain": [
        "FS00: define tensors + alignment score",
        "FS01: rule-based image to text",
        "FS02: learned CNN image to label-word",
    ],
}
(RES / "summary_fs00_fs02.json").write_text(json.dumps(summary, indent=2))
(OUT / "SUCCESS").write_text("ok\n")
print(json.dumps(summary, indent=2))
print("ALL FS00-FS02 COMPLETE")
